# 01 — Exploratory Data Analysis
**DSC 465 | Blackjack Optimal Action ML**

Goals:
1. Understand the dataset schema
2. Inspect distributions of key variables
3. Identify data quality issues
4. Document column meanings for use in feature engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

DATA_PATH = Path('../data/blackjack_simulator.csv')
assert DATA_PATH.exists(), f'Dataset not found at {DATA_PATH}'

## 1. Load and Inspect Schema

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns ({len(df.columns)}):\n{list(df.columns)}')

In [ ]:
df.head(5)

In [ ]:
df.dtypes

In [ ]:
df.describe(include='all').T

## 2. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct}).query('missing_count > 0')

## 3. Game / Hand Structure

In [ ]:
# Replace 'game_id' with the actual game identifier column name
# After running detect_schema() in train.py, check what it finds
game_col = [c for c in df.columns if 'game' in c.lower() or 'id' in c.lower()]
print('Potential game ID columns:', game_col)

if game_col:
    g = df[game_col[0]]
    print(f'\nUnique games: {g.nunique():,}')
    print(f'Rows per game (avg): {len(df) / g.nunique():.1f}')
    print(f'Rows per game (value counts):\n{df.groupby(game_col[0]).size().value_counts().head(10)}')

## 4. Action Distribution

In [ ]:
# Replace 'action' with the actual action column name
action_col = [c for c in df.columns if 'action' in c.lower() or 'move' in c.lower() or 'play' in c.lower()]
print('Potential action columns:', action_col)

if action_col:
    act = df[action_col[0]]
    print(f'\nUnique values: {act.unique()}')
    print(f'\nValue counts:\n{act.value_counts()}')
    
    fig, ax = plt.subplots(figsize=(6, 4))
    act.value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Action Distribution')
    ax.set_xlabel('Action')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()

## 5. Outcome Distribution

In [ ]:
# Check for outcome / win / loss columns
outcome_cols = [c for c in df.columns 
                if any(kw in c.lower() for kw in ['win', 'loss', 'bust', 'payout', 'result', 'outcome', 'blackjack'])]
print('Potential outcome columns:', outcome_cols)

for col in outcome_cols[:5]:
    print(f'\n{col}:\n{df[col].value_counts()}')

## 6. Card Columns

In [ ]:
card_cols = [c for c in df.columns if 'card' in c.lower()]
print('Card columns:', card_cols)

for col in card_cols[:3]:
    print(f'\n{col} unique values: {sorted(df[col].dropna().unique())}')

## 7. Player Total Distribution

In [ ]:
total_cols = [c for c in df.columns if 'total' in c.lower() or 'sum' in c.lower()]
print('Total columns:', total_cols)

if total_cols:
    fig, axes = plt.subplots(1, min(2, len(total_cols)), figsize=(12, 4))
    if len(total_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, total_cols[:2]):
        df[col].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
        ax.set_title(col)
    plt.tight_layout()
    plt.show()

## 8. Action vs. Outcome Crosstab

In [ ]:
# Requires knowing the action and win columns — update names below
# action_col_name = 'action'    # <-- update this
# win_col_name    = 'player_win' # <-- update this

# Uncomment and run after confirming column names:
# ct = pd.crosstab(df[action_col_name], df[win_col_name], normalize='index')
# print(ct)

## 9. Dealer Upcard Distribution

In [ ]:
dealer_cols = [c for c in df.columns if 'dealer' in c.lower()]
print('Dealer columns:', dealer_cols)

if dealer_cols:
    upcard_col = dealer_cols[0]
    fig, ax = plt.subplots(figsize=(7, 4))
    df[upcard_col].value_counts().sort_index().plot(kind='bar', ax=ax, color='darkorange', edgecolor='white')
    ax.set_title(f'Dealer column: {upcard_col}')
    plt.tight_layout()
    plt.show()

## 10. Schema Summary

After running the cells above, fill in this table to document the confirmed column roles:

| Role | Column name | Notes |
|---|---|---|
| Game ID | ??? | unique identifier per game (150K games) |
| Player position | ??? | seat / player index within game |
| Action | ??? | hit / stand / double / split |
| Dealer upcard | ??? | dealer's visible card |
| Player cards | ??? | all player card columns |
| Win flag | ??? | 1 = player won |
| Bust flag | ??? | 1 = player busted |
| Blackjack | ??? | 1 = natural blackjack |
| Payout | ??? | numeric payout (if exists) |

**Update `detect_schema()` in `src/train.py` if auto-detection misses any of these.**

In [ ]:
# Quick sanity: 900K rows, 150K unique games, 6 players per game
print(f'Total rows: {len(df):,}')
print(f'Expected: 900,000')